# Reddit Historical Data Scraper

Este notebook permite obtener datos históricos de Reddit del último año de forma controlada y por batches.


In [1]:
import os
import sys
import json
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import time
import logging

# Agregar el directorio raíz al path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.scraping.scraping_reddit import RedditScraper
from src.scraping.reddit_config import get_reddit_config


ModuleNotFoundError: No module named 'src.scraping.scraping_reddit'

## Configuración Inicial


In [ ]:
# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/historical_scraper.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Crear directorios necesarios
Path('data/historical').mkdir(parents=True, exist_ok=True)
Path('logs').mkdir(exist_ok=True)

print("✅ Configuración inicial completada")


## Definir Períodos de Scraping


In [ ]:
def create_monthly_batches(start_date, end_date):
    """Crear batches mensuales para el scraping histórico."""
    batches = []
    current = start_date
    
    while current <= end_date:
        # Primer día del mes
        month_start = current.replace(day=1)
        
        # Último día del mes
        if current.month == 12:
            month_end = current.replace(year=current.year + 1, month=1, day=1) - timedelta(days=1)
        else:
            month_end = current.replace(month=current.month + 1, day=1) - timedelta(days=1)
        
        # Ajustar al rango solicitado
        month_start = max(month_start, start_date)
        month_end = min(month_end, end_date)
        
        batches.append({
            'start': month_start,
            'end': month_end,
            'name': f"{month_start.strftime('%Y-%m')}"
        })
        
        # Siguiente mes
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1)
        else:
            current = current.replace(month=current.month + 1)
    
    return batches

# Definir rango de fechas (último año)
end_date = datetime.now()
start_date = end_date - timedelta(days=365)

print(f"📅 Rango de fechas: {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}")

# Crear batches mensuales
batches = create_monthly_batches(start_date, end_date)
print(f"📊 Total de batches: {len(batches)}")

# Mostrar los primeros 3 batches
for i, batch in enumerate(batches[:3]):
    print(f"  {i+1}. {batch['name']}: {batch['start'].strftime('%Y-%m-%d')} a {batch['end'].strftime('%Y-%m-%d')}")


## Configurar Scraper


In [ ]:
# Inicializar scraper
try:
    scraper = RedditScraper()
    print("✅ Scraper inicializado correctamente")
except Exception as e:
    print(f"❌ Error inicializando scraper: {e}")
    print("Verifica que el archivo .env esté configurado correctamente")


## Función de Scraping por Batch


In [ ]:
def scrape_historical_batch(batch_info, max_posts_per_subreddit=1000):
    """Scrapear datos históricos para un batch específico."""
    
    batch_name = batch_info['name']
    start_date = batch_info['start']
    end_date = batch_info['end']
    
    print(f"\n🔄 Procesando batch: {batch_name}")
    print(f"📅 Período: {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}")
    
    try:
        # Scrapear todos los subreddits para este período
        all_data = {}
        
        for subreddit in scraper.subreddits:
            print(f"  📊 Scrapeando r/{subreddit}...")
            
            # Scrapear posts
            posts = scraper.scrape_subreddit_posts(
                subreddit, 
                limit=max_posts_per_subreddit,
                time_filter='all'  # Para datos históricos
            )
            
            # Scrapear comentarios
            comments = scraper.scrape_subreddit_comments(
                subreddit, 
                limit=max_posts_per_subreddit,
                time_filter='all'
            )
            
            # Filtrar por fecha
            filtered_posts = [
                post for post in posts 
                if start_date <= datetime.fromtimestamp(post['created_utc']) <= end_date
            ]
            
            filtered_comments = [
                comment for comment in comments 
                if start_date <= datetime.fromtimestamp(comment['created_utc']) <= end_date
            ]
            
            all_data[subreddit] = {
                'posts': filtered_posts,
                'comments': filtered_comments,
                'total_posts': len(filtered_posts),
                'total_comments': len(filtered_comments)
            }
            
            print(f"    ✅ {len(filtered_posts)} posts, {len(filtered_comments)} comentarios")
            
            # Pausa para evitar rate limits
            time.sleep(2)
        
        # Guardar datos del batch
        batch_filename = f"data/historical/reddit_historical_{batch_name}.json"
        with open(batch_filename, 'w', encoding='utf-8') as f:
            json.dump(all_data, f, indent=2, ensure_ascii=False, default=str)
        
        # Guardar también en CSV
        csv_base_filename = f"data/historical/reddit_historical_{batch_name}"
        scraper.save_to_csv(all_data, csv_base_filename)
        
        # Resumen del batch
        total_posts = sum(data['total_posts'] for data in all_data.values())
        total_comments = sum(data['total_comments'] for data in all_data.values())
        
        print(f"✅ Batch {batch_name} completado:")
        print(f"  📝 Total posts: {total_posts}")
        print(f"  💬 Total comments: {total_comments}")
        print(f"  💾 Archivos guardados: {batch_filename}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error en batch {batch_name}: {e}")
        return False


## Ejecutar Scraping Histórico


In [ ]:
# Configurar parámetros
MAX_POSTS_PER_SUBREDDIT = 1000  # Ajustar según necesidades
BATCH_DELAY = 30  # Segundos entre batches

print(f"🚀 Iniciando scraping histórico")
print(f"📊 Máximo posts por subreddit: {MAX_POSTS_PER_SUBREDDIT}")
print(f"⏱️ Delay entre batches: {BATCH_DELAY} segundos")
print(f"📅 Total batches: {len(batches)}")

# Estadísticas
successful_batches = 0
failed_batches = 0
start_time = datetime.now()

for i, batch in enumerate(batches):
    print(f"\n{'='*60}")
    print(f"📊 Procesando batch {i+1}/{len(batches)}: {batch['name']}")
    
    success = scrape_historical_batch(batch, MAX_POSTS_PER_SUBREDDIT)
    
    if success:
        successful_batches += 1
    else:
        failed_batches += 1
    
    # Pausa entre batches (excepto en el último)
    if i < len(batches) - 1:
        print(f"⏳ Esperando {BATCH_DELAY} segundos antes del siguiente batch...")
        time.sleep(BATCH_DELAY)

# Resumen final
end_time = datetime.now()
duration = end_time - start_time

print(f"\n{'='*60}")
print(f"🎉 Scraping histórico completado!")
print(f"⏱️ Duración total: {duration}")
print(f"✅ Batches exitosos: {successful_batches}")
print(f"❌ Batches fallidos: {failed_batches}")
print(f"📊 Tasa de éxito: {successful_batches/len(batches)*100:.1f}%")


## Combinar Datos Históricos


In [ ]:
def combine_historical_data():
    """Combinar todos los datos históricos en un solo archivo."""
    
    print("🔄 Combinando datos históricos...")
    
    # Buscar todos los archivos históricos
    historical_files = list(Path('data/historical').glob('reddit_historical_*.json'))
    
    if not historical_files:
        print("❌ No se encontraron archivos históricos")
        return
    
    print(f"📊 Encontrados {len(historical_files)} archivos históricos")
    
    # Combinar datos
    combined_data = {}
    
    for file_path in sorted(historical_files):
        print(f"  📄 Procesando {file_path.name}...")
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                batch_data = json.load(f)
            
            # Combinar por subreddit
            for subreddit, data in batch_data.items():
                if subreddit not in combined_data:
                    combined_data[subreddit] = {
                        'posts': [],
                        'comments': [],
                        'total_posts': 0,
                        'total_comments': 0
                    }
                
                combined_data[subreddit]['posts'].extend(data['posts'])
                combined_data[subreddit]['comments'].extend(data['comments'])
                combined_data[subreddit]['total_posts'] += data['total_posts']
                combined_data[subreddit]['total_comments'] += data['total_comments']
                
        except Exception as e:
            print(f"⚠️ Error procesando {file_path.name}: {e}")
    
    # Guardar datos combinados
    combined_filename = 'data/historical/reddit_historical_combined.json'
    with open(combined_filename, 'w', encoding='utf-8') as f:
        json.dump(combined_data, f, indent=2, ensure_ascii=False, default=str)
    
    # Guardar también en CSV
    csv_base_filename = 'data/historical/reddit_historical_combined'
    scraper.save_to_csv(combined_data, csv_base_filename)
    
    # Resumen final
    total_posts = sum(data['total_posts'] for data in combined_data.values())
    total_comments = sum(data['total_comments'] for data in combined_data.values())
    
    print(f"\n✅ Datos históricos combinados:")
    print(f"📊 Total posts: {total_posts}")
    print(f"💬 Total comments: {total_comments}")
    print(f"💾 Archivo combinado: {combined_filename}")

# Ejecutar combinación
combine_historical_data()


## Análisis de Datos Históricos


In [ ]:
# Cargar datos combinados para análisis
try:
    with open('data/historical/reddit_historical_combined.json', 'r', encoding='utf-8') as f:
        historical_data = json.load(f)
    
    print("📊 Análisis de datos históricos:")
    print("=" * 40)
    
    for subreddit, data in historical_data.items():
        print(f"\n📈 r/{subreddit}:")
        print(f"  📝 Posts: {data['total_posts']}")
        print(f"  💬 Comentarios: {data['total_comments']}")
        
        # Análisis temporal si hay datos
        if data['posts']:
            # Convertir timestamps a fechas
            post_dates = [datetime.fromtimestamp(post['created_utc']) for post in data['posts']]
            
            if post_dates:
                earliest = min(post_dates)
                latest = max(post_dates)
                print(f"  📅 Rango temporal: {earliest.strftime('%Y-%m-%d')} a {latest.strftime('%Y-%m-%d')}")
    
    # Resumen general
    total_posts = sum(data['total_posts'] for data in historical_data.values())
    total_comments = sum(data['total_comments'] for data in historical_data.values())
    
    print(f"\n🎯 Resumen general:")
    print(f"📊 Total posts: {total_posts}")
    print(f"💬 Total comments: {total_comments}")
    print(f"📈 Total subreddits: {len(historical_data)}")
    
except FileNotFoundError:
    print("❌ No se encontraron datos históricos combinados")
    print("Ejecuta primero la celda de combinación de datos")
